# L8: 错误处理与重试

> 掌握 LLM API 调用中的错误处理和重试策略

In [1]:
# === 环境设置 ===
import sys
import os
import asyncio
import random
from pathlib import Path

# 自动找到项目根目录
current_path = Path(os.getcwd())
project_path = None

for path in [current_path] + list(current_path.parents):
    if (path / "backend").exists():
        project_path = str(path)
        break

if project_path and project_path not in sys.path:
    sys.path.insert(0, project_path)

print(f"项目路径: {project_path}")
print(f"Python 版本: {sys.version.split()[0]}")

# 验证模块导入
try:
    from backend.llm import LLMFactory, Message
    from backend.llm.base import LLMError, LLMRateLimitError
    print("✓ LLM 模块导入成功")
except ImportError as e:
    print(f"✗ 模块导入失败: {e}")

项目路径: c:\Users\zying\Desktop\pro_agent\agent-learning-project
Python 版本: 3.14.4
✓ LLM 模块导入成功


## 8.1 为什么需要错误处理？

### LLM API 调用的不确定性

| 问题类型 | 发生概率 | 影响 |
|----------|----------|------|
| **网络问题** | 中等 | 请求超时、连接失败 |
| **API 限流** | 高（大量请求）| 429 错误 |
| **服务不可用** | 低 | 500/503 错误 |
| **认证失败** | 低（配置后）| 401 错误 |
| **参数错误** | 低（开发时）| 400 错误 |

### 没有错误处理的后果

```
用户体验:
  → 程序崩溃
  → 无错误提示
  → 数据丢失

开发者体验:
  → 难以定位问题
  → 无法统计成功率
  → 无法监控告警
```

## 8.2 错误类型详解

### 项目中的错误类层次

In [2]:
from backend.llm.base import (
    LLMError,
    LLMRateLimitError,
    LLMTimeoutError,
    LLMAuthenticationError,
    LLMInvalidRequestError
)

print("=== LLM 错误类层次 ===\n")
print("LLMError (基础类)")
print("  ├─ LLMRateLimitError      → 429 Too Many Requests")
print("  ├─ LLMTimeoutError        → 连接/读取超时")
print("  ├─ LLMAuthenticationError → 401 Unauthorized")
print("  └─ LLMInvalidRequestError → 400 Bad Request")

print("\n=== 错误属性 ===\n")

# 创建一个示例错误
error = LLMError(
    message="API 调用失败",
    provider="DeepSeek",
    details={"status_code": 500, "reason": "Internal Server Error"}
)

print(f"错误消息: {error.message}")
print(f"提供商: {error.provider}")
print(f"详细信息: {error.details}")

=== LLM 错误类层次 ===

LLMError (基础类)
  ├─ LLMRateLimitError      → 429 Too Many Requests
  ├─ LLMTimeoutError        → 连接/读取超时
  ├─ LLMAuthenticationError → 401 Unauthorized
  └─ LLMInvalidRequestError → 400 Bad Request

=== 错误属性 ===

错误消息: API 调用失败
提供商: DeepSeek
详细信息: {'status_code': 500, 'reason': 'Internal Server Error'}


### HTTP 状态码与错误类型映射

In [3]:
# 错误映射表
error_mapping = [
    ("401", "LLMAuthenticationError", "API Key 无效或过期"),
    ("403", "LLMAuthenticationError", "无权限访问此资源"),
    ("404", "LLMInvalidRequestError", "模型或端点不存在"),
    ("429", "LLMRateLimitError", "请求过于频繁，触发限流"),
    ("500", "LLMError", "服务器内部错误"),
    ("502", "LLMError", "网关错误"),
    ("503", "LLMError", "服务暂时不可用"),
]

print("=== HTTP 状态码 → 错误类型 ===\n")
print(f"{'状态码':<10} {'错误类型':<30} {'说明'}")
print("-" * 70)
for code, error_type, desc in error_mapping:
    print(f"{code:<10} {error_type:<30} {desc}")

=== HTTP 状态码 → 错误类型 ===

状态码        错误类型                           说明
----------------------------------------------------------------------
401        LLMAuthenticationError         API Key 无效或过期
403        LLMAuthenticationError         无权限访问此资源
404        LLMInvalidRequestError         模型或端点不存在
429        LLMRateLimitError              请求过于频繁，触发限流
500        LLMError                       服务器内部错误
502        LLMError                       网关错误
503        LLMError                       服务暂时不可用


## 8.3 基础错误处理

In [10]:
async def basic_error_handling():
    """基础错误处理示例"""
    from backend.llm import Message
    
    # 使用无效 API Key 演示
    llm = LLMFactory.create("deepseek", api_key="invalid_key")
    
    try:
        response = await llm.chat([
            Message(role="user", content="你好")
        ])
        return response
        
    except LLMAuthenticationError as e:
        print(f"❌ 认证失败: {e.message}")
        print(f"   提供商: {e.provider}")
        print(f"   建议: 请检查 API Key 是否正确")
        
    except LLMRateLimitError as e:
        print(f"❌ 速率限制: {e.message}")
        print(f"   建议: 请等待一段时间后重试")
        
    except LLMTimeoutError as e:
        print(f"❌ 请求超时: {e.message}")
        print(f"   建议: 检查网络连接或增加超时时间")
        
    except LLMInvalidRequestError as e:
        print(f"❌ 无效请求: {e.message}")
        print(f"   详情: {e.details}")
        
    except LLMError as e:
        print(f"❌ LLM 错误: {e.message}")
        print(f"   提供商: {e.provider}")
        
    except Exception as e:
        print(f"❌ 未知错误: {type(e).__name__}: {e}")

# 运行（需要取消注释）
await basic_error_handling()

2026-05-23 01:47:45,115 - httpx - INFO - HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 401 Authorization Required"


❌ 认证失败: DeepSeek API认证失败，请检查API密钥
   提供商: DeepSeek
   建议: 请检查 API Key 是否正确


## 8.4 重试策略

### 为什么需要重试？

| 错误类型 | 是否可重试 | 说明 |
|----------|-----------|------|
| 网络超时 | ✅ | 临时性问题，重试可能成功 |
| 速率限制 | ✅ | 等待后重试 |
| 5xx 错误 | ✅ | 服务器临时问题 |
| 4xx 错误 | ❌ | 客户端错误，重试无意义 |

### 重试策略对比

In [5]:
import asyncio

# 模拟不同的重试策略

async def fixed_delay_retry(attempts=3, delay=1):
    """固定延迟重试"""
    for i in range(attempts):
        try:
            # 模拟可能失败的操作
            if random.random() < 0.5:
                raise Exception("随机失败")
            print(f"✓ 第 {i+1} 次尝试成功")
            return True
        except Exception as e:
            if i < attempts - 1:
                print(f"第 {i+1} 次失败，{delay}秒后重试...")
                await asyncio.sleep(delay)
    print("全部尝试失败")
    return False

async def exponential_backoff_retry(attempts=3, base_delay=1):
    """指数退避重试"""
    for i in range(attempts):
        try:
            if random.random() < 0.5:
                raise Exception("随机失败")
            print(f"✓ 第 {i+1} 次尝试成功")
            return True
        except Exception as e:
            if i < attempts - 1:
                delay = base_delay * (2 ** i)
                print(f"第 {i+1} 次失败，{delay}秒后重试...")
                await asyncio.sleep(delay)
    print("全部尝试失败")
    return False

async def jitter_retry(attempts=3, base_delay=1, jitter_range=0.5):
    """带抖动的重试（避免惊群效应）"""
    for i in range(attempts):
        try:
            if random.random() < 0.5:
                raise Exception("随机失败")
            print(f"✓ 第 {i+1} 次尝试成功")
            return True
        except Exception as e:
            if i < attempts - 1:
                delay = base_delay * (2 ** i) + random.uniform(-jitter_range, jitter_range)
                delay = max(0, delay)
                print(f"第 {i+1} 次失败，{delay:.2f}秒后重试...")
                await asyncio.sleep(delay)
    print("全部尝试失败")
    return False

# 对比三种策略
print("=== 重试策略对比 ===\n")

print("1. 固定延迟 (1秒)")
print("   延迟: 1s, 1s, 1s")
print("   优点: 简单")
print("   缺点: 可能不够灵活\n")

print("2. 指数退避")
print("   延迟: 1s, 2s, 4s")
print("   优点: 适合网络恢复")
print("   缺点: 多客户端可能同时重试\n")

print("3. 指数退避 + 抖动")
print("   延迟: 1s±0.5, 2s±0.5, 4s±0.5")
print("   优点: 避免惊群效应")
print("   缺点: 稍复杂\n")

=== 重试策略对比 ===

1. 固定延迟 (1秒)
   延迟: 1s, 1s, 1s
   优点: 简单
   缺点: 可能不够灵活

2. 指数退避
   延迟: 1s, 2s, 4s
   优点: 适合网络恢复
   缺点: 多客户端可能同时重试

3. 指数退避 + 抖动
   延迟: 1s±0.5, 2s±0.5, 4s±0.5
   优点: 避免惊群效应
   缺点: 稍复杂



### 实现一个通用的重试装饰器

In [6]:
import functools
import time
from typing import Callable, Type, Tuple, Optional

def retry_on_exception(
    exceptions: Tuple[Type[Exception], ...] = (Exception,),
    max_attempts: int = 3,
    base_delay: float = 1,
    exponential: bool = True,
    jitter: bool = True,
):
    """
    通用重试装饰器
    
    Args:
        exceptions: 需要重试的异常类型
        max_attempts: 最大尝试次数
        base_delay: 基础延迟时间（秒）
        exponential: 是否使用指数退避
        jitter: 是否添加随机抖动
    """
    def decorator(func: Callable):
        @functools.wraps(func)
        async def async_wrapper(*args, **kwargs):
            last_exception = None
            
            for attempt in range(max_attempts):
                try:
                    return await func(*args, **kwargs)
                except exceptions as e:
                    last_exception = e
                    
                    if attempt == max_attempts - 1:
                        break
                    
                    # 计算延迟
                    if exponential:
                        delay = base_delay * (2 ** attempt)
                    else:
                        delay = base_delay
                    
                    # 添加抖动
                    if jitter:
                        delay = delay * (0.5 + random.random())
                    
                    print(f"[重试] 第 {attempt + 1}/{max_attempts} 次失败，{delay:.2f}秒后重试: {e}")
                    await asyncio.sleep(delay)
            
            raise last_exception
        
        return async_wrapper
    
    return decorator

# 使用示例
@retry_on_exception(
    exceptions=(LLMRateLimitError, LLMTimeoutError),
    max_attempts=3,
    base_delay=1,
    exponential=True,
    jitter=True,
)
async def call_llm_with_retry(llm, messages):
    """带重试的 LLM 调用"""
    return await llm.chat(messages)

print("重试装饰器已定义！")
print("\n使用方法:")
print("@retry_on_exception(max_attempts=3, base_delay=1)")
print("async def my_function():")
print("    ...")

重试装饰器已定义！

使用方法:
@retry_on_exception(max_attempts=3, base_delay=1)
async def my_function():
    ...


## 8.5 断路器模式

### 什么是断路器？

断路器模式可以防止系统持续调用失败的服务，避免雪崩效应。

In [7]:
from enum import Enum
from datetime import datetime, timedelta

class CircuitState(Enum):
    CLOSED = "closed"       # 正常状态
    OPEN = "open"           # 断开状态（不请求）
    HALF_OPEN = "half_open" # 半开状态（尝试恢复）

class CircuitBreaker:
    """
    断路器实现
    
    状态转换:
    CLOSED → OPEN: 失败次数超过阈值
    OPEN → HALF_OPEN: 超过恢复时间
    HALF_OPEN → CLOSED: 尝试成功
    HALF_OPEN → OPEN: 尝试失败
    """
    
    def __init__(
        self,
        failure_threshold: int = 5,
        recovery_timeout: int = 60,
    ):
        self.failure_threshold = failure_threshold
        self.recovery_timeout = recovery_timeout
        
        self._state = CircuitState.CLOSED
        self._failure_count = 0
        self._last_failure_time = None
        self._success_count = 0
    
    @property
    def state(self) -> CircuitState:
        """获取当前状态"""
        # 检查是否可以从 OPEN 转到 HALF_OPEN
        if self._state == CircuitState.OPEN:
            if (datetime.now() - self._last_failure_time).total_seconds() > self.recovery_timeout:
                self._state = CircuitState.HALF_OPEN
        return self._state
    
    def record_success(self):
        """记录成功"""
        self._failure_count = 0
        if self._state == CircuitState.HALF_OPEN:
            self._state = CircuitState.CLOSED
        self._success_count += 1
    
    def record_failure(self):
        """记录失败"""
        self._failure_count += 1
        self._last_failure_time = datetime.now()
        
        if self._failure_count >= self.failure_threshold:
            self._state = CircuitState.OPEN
    
    def can_execute(self) -> bool:
        """是否可以执行请求"""
        return self.state != CircuitState.OPEN
    
    def get_stats(self) -> dict:
        """获取统计信息"""
        return {
            "state": self.state.value,
            "failure_count": self._failure_count,
            "success_count": self._success_count,
            "threshold": self.failure_threshold,
        }

# 演示断路器
async def demo_circuit_breaker():
    """断路器演示"""
    cb = CircuitBreaker(failure_threshold=3, recovery_timeout=5)
    
    print("=== 断路器演示 ===\n")
    
    # 模拟连续失败
    for i in range(5):
        if cb.can_execute():
            print(f"[请求 {i+1}] 状态: {cb.state.value}, 允许执行")
            # 模拟失败
            cb.record_failure()
            print(f"  → 记录失败，失败次数: {cb._failure_count}")
        else:
            print(f"[请求 {i+1}] 状态: {cb.state.value}, 拒绝执行（断路器打开）")
    
    print(f"\n统计: {cb.get_stats()}")

await demo_circuit_breaker()

=== 断路器演示 ===

[请求 1] 状态: closed, 允许执行
  → 记录失败，失败次数: 1
[请求 2] 状态: closed, 允许执行
  → 记录失败，失败次数: 2
[请求 3] 状态: closed, 允许执行
  → 记录失败，失败次数: 3
[请求 4] 状态: open, 拒绝执行（断路器打开）
[请求 5] 状态: open, 拒绝执行（断路器打开）

统计: {'state': 'open', 'failure_count': 3, 'success_count': 0, 'threshold': 3}


## 8.6 综合错误处理方案

### 完整的错误处理流程

In [8]:
class RobustLLMClient:
    """
    健壮的 LLM 客户端包装器
    
    功能:
    - 自动重试（指数退避）
    - 断路器保护
    - 降级处理
    - 错误日志
    """
    
    def __init__(
        self,
        llm,
        fallback_llm=None,
        max_retries: int = 3,
        circuit_breaker_threshold: int = 5,
    ):
        self.llm = llm
        self.fallback_llm = fallback_llm
        self.circuit_breaker = CircuitBreaker(failure_threshold=circuit_breaker_threshold)
        self.max_retries = max_retries
        self.stats = {"success": 0, "failure": 0, "fallback": 0}
    
    async def chat(self, messages, **kwargs):
        """带完整错误处理的 chat 方法"""
        # 检查断路器
        if not self.circuit_breaker.can_execute():
            if self.fallback_llm:
                print("[降级] 主服务不可用，使用备用服务")
                self.stats["fallback"] += 1
                return await self._chat_with_retry(self.fallback_llm, messages, **kwargs)
            else:
                raise LLMError("服务不可用且无备用服务")
        
        try:
            response = await self._chat_with_retry(self.llm, messages, **kwargs)
            self.circuit_breaker.record_success()
            self.stats["success"] += 1
            return response
        except Exception as e:
            self.circuit_breaker.record_failure()
            self.stats["failure"] += 1
            
            # 尝试降级
            if self.fallback_llm:
                print(f"[降级] 主服务失败: {e}，使用备用服务")
                self.stats["fallback"] += 1
                return await self._chat_with_retry(self.fallback_llm, messages, **kwargs)
            
            raise
    
    async def _chat_with_retry(self, llm, messages, **kwargs):
        """带重试的 chat 调用"""
        last_exception = None
        
        for attempt in range(self.max_retries):
            try:
                return await llm.chat(messages, **kwargs)
            except (LLMRateLimitError, LLMTimeoutError) as e:
                last_exception = e
                if attempt < self.max_retries - 1:
                    delay = 1 * (2 ** attempt)
                    print(f"[重试] 第 {attempt + 1}/{self.max_retries} 次，{delay}秒后重试")
                    await asyncio.sleep(delay)
        
        raise last_exception
    
    def get_stats(self):
        """获取统计信息"""
        return {
            **self.stats,
            "circuit_breaker": self.circuit_breaker.get_stats(),
        }

print("RobustLLMClient 类已定义！")
print("\n特性:")
print("  ✓ 自动重试（指数退避）")
print("  ✓ 断路器保护")
print("  ✓ 服务降级（fallback）")
print("  ✓ 统计信息")

RobustLLMClient 类已定义！

特性:
  ✓ 自动重试（指数退避）
  ✓ 断路器保护
  ✓ 服务降级（fallback）
  ✓ 统计信息


## 8.7 日志与监控

In [9]:
import logging
from datetime import datetime

# 配置日志
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)

class MonitoredLLMClient:
    """带监控的 LLM 客户端"""
    
    def __init__(self, llm):
        self.llm = llm
        self.logger = logging.getLogger(f"LLM.{id(llm)}")
        self.metrics = {
            "requests": 0,
            "errors": 0,
            "total_tokens": 0,
            "total_latency_ms": 0,
        }
    
    async def chat(self, messages, **kwargs):
        """带监控的 chat 方法"""
        start_time = datetime.now()
        self.metrics["requests"] += 1
        
        try:
            self.logger.info(f"发送请求，消息数: {len(messages)}")
            
            response = await self.llm.chat(messages, **kwargs)
            
            # 记录指标
            latency = (datetime.now() - start_time).total_seconds() * 1000
            self.metrics["total_latency_ms"] += latency
            self.metrics["total_tokens"] += response.usage.total_tokens
            
            self.logger.info(
                f"请求成功 | 延迟: {latency:.0f}ms | "
                f"Tokens: {response.usage.total_tokens}"
            )
            
            return response
            
        except Exception as e:
            self.metrics["errors"] += 1
            self.logger.error(f"请求失败: {e}")
            raise
    
    def get_metrics(self):
        """获取监控指标"""
        requests = self.metrics["requests"]
        return {
            "total_requests": requests,
            "total_errors": self.metrics["errors"],
            "error_rate": self.metrics["errors"] / requests if requests > 0 else 0,
            "total_tokens": self.metrics["total_tokens"],
            "avg_latency_ms": self.metrics["total_latency_ms"] / requests if requests > 0 else 0,
        }

print("MonitoredLLMClient 类已定义！")
print("\n监控指标:")
print("  - 请求总数")
print("  - 错误总数")
print("  - 错误率")
print("  - Token 总数")
print("  - 平均延迟")

MonitoredLLMClient 类已定义！

监控指标:
  - 请求总数
  - 错误总数
  - 错误率
  - Token 总数
  - 平均延迟


## 8.8 常见面试问题

**Q: 什么是指数退避？为什么使用它？**
- A:
  - 指数退避：每次重试延迟时间翻倍（1s, 2s, 4s, 8s...）
  - 优点：
    - 给服务更多恢复时间
    - 减少重试压力
    - 适合网络恢复场景

**Q: 什么是断路器模式？**
- A:
  - 三种状态：CLOSED（正常）、OPEN（断开）、HALF_OPEN（半开）
  - 作用：防止持续调用失败服务
  - 状态转换：
    - CLOSED → OPEN: 失败次数超阈值
    - OPEN → HALF_OPEN: 超过恢复时间
    - HALF_OPEN → CLOSED: 尝试成功

**Q: 哪些错误应该重试？哪些不应该？**
- A:
  | 错误类型 | 是否重试 | 原因 |
  |----------|----------|------|
  | 网络超时 | ✅ | 临时问题 |
  | 速率限制(429) | ✅ | 等待后可恢复 |
  | 5xx错误 | ✅ | 服务临时故障 |
  | 401认证失败 | ❌ | 重试无意义 |
  | 400参数错误 | ❌ | 需要修改参数 |

**Q: 什么是服务降级？**
- A:
  - 当主服务不可用时，使用备用方案
  - 示例：
    - DeepSeek 不可用 → 使用 OpenAI
    - GPT-4 不可用 → 降级到 GPT-3.5
    - 在线服务不可用 → 使用本地模型

**Q: 如何避免惊群效应（Thundering Herd）？**
- A:
  - 添加随机抖动（Jitter）
  - 让不同客户端的重试时间错开
  - 实现：delay = base_delay * (2^attempt) + random(-jitter, +jitter)

**Q: 如何监控 LLM API 调用？**
- A:
  - 关键指标：
    - 请求总数/错误数
    - 错误率
    - 平均延迟/P99 延迟
    - Token 使用量
    - 成本
  - 告警规则：
    - 错误率 > 5%
    - 延迟 > 5s
    - 成本异常

---

## 总结

本课程学习了 LLM API 调用的错误处理和重试机制：

| 知识点 | 说明 |
|--------|------|
| **错误类型** | 认证、限流、超时、参数错误 |
| **重试策略** | 固定延迟、指数退避、抖动 |
| **断路器模式** | 三状态保护机制 |
| **服务降级** | 主备切换 |
| **监控日志** | 指标收集和告警 |
| **综合方案** | 重试+断路器+降级+监控 |

**下一步**: L9 将学习 Function Calling！